# 04 — IBM Quantum Hardware Benchmarks

This notebook executes the Adaptive_QEM_IBM benchmark suite on **real IBM Quantum hardware** and records the hardware conditions required for reproducible analysis. It collects backend metadata, calibration information, transpiled circuit metrics, job IDs, raw measurement counts, and benchmark performance.

**Important:** No IBM credentials or API tokens are stored in this notebook. IBM Quantum authentication must already be configured in the local Qiskit environment.

## Experimental Flow

**Benchmark circuit → IBM backend selection → calibration snapshot → transpilation → hardware execution → raw counts → hardware metrics → saved dataset**

The resulting dataset becomes the hardware baseline for `05_qem_analysis.ipynb`.

In [ ]:
from pathlib import Path
import sys
import json
import math
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "hardware"
CALIBRATION_DIR = PROJECT_ROOT / "data" / "calibration"
RESULTS_DIR = PROJECT_ROOT / "results" / "tables"

for directory in [DATA_DIR, CALIBRATION_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SHOTS = 4096
OPTIMIZATION_LEVEL = 3
SEED_TRANSPILER = 42

print("Project root:", PROJECT_ROOT)
print("Hardware data:", DATA_DIR)

## 1. IBM Quantum Runtime Connection

The preferred workflow is to authenticate once using the local Qiskit IBM Runtime account. This notebook does not store tokens or API keys.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService

try:
    service = QiskitRuntimeService()
    print("IBM Quantum Runtime connection: OK")
except Exception as exc:
    service = None
    print("IBM Quantum Runtime connection failed.")
    print("Reason:", exc)
    print("\nConfigure IBM Quantum authentication before hardware execution.")

## 2. Select the Experimental Backend

Set the backend explicitly for a reproducible study. Change only this variable when moving the experiment to another IBM backend.

In [ ]:
BACKEND_NAME = "ibm_kingston"

backend = None

if service is not None:
    try:
        backend = service.backend(BACKEND_NAME)
        print("Backend:", backend.name)
        print("Number of qubits:", backend.num_qubits)
        print("Backend status:", backend.status())
    except Exception as exc:
        print(f"Could not access {BACKEND_NAME}:")
        print(exc)
else:
    print("Backend selection skipped because Runtime is unavailable.")

## 3. Load the Benchmark Circuits

The same circuits used in the ideal and controlled-noise stages are reused here. This keeps the experimental comparison consistent.

In [ ]:
from circuits.bell import bell_phi_plus
from circuits.ghz import create_ghz
from circuits.teleportation import teleportation
from circuits.superdense import superdense_coding
from circuits.qft import qft
from circuits.grover import grover_2qubit
from circuits.qaoa import qaoa_two_node
from circuits.qpe import qpe
from circuits.qrng import qrng

benchmarks = {
    "Bell_Phi_Plus": bell_phi_plus(),
    "GHZ_3": create_ghz(3),
    "GHZ_4": create_ghz(4),
    "GHZ_5": create_ghz(5),
    "Teleportation": teleportation(),
    "Superdense_00": superdense_coding("00"),
    "Superdense_01": superdense_coding("01"),
    "Superdense_10": superdense_coding("10"),
    "Superdense_11": superdense_coding("11"),
    "QFT_3": qft(3),
    "QFT_4": qft(4),
    "Grover_2Q": grover_2qubit(),
    "QAOA_2Q": qaoa_two_node(gamma=math.pi / 4, beta=math.pi / 8),
    "QPE": qpe(),
    "QRNG_4": qrng(4),
}

print(f"Loaded {len(benchmarks)} benchmark circuits.")

## 4. Capture Backend Metadata

The metadata snapshot records the backend identity and configuration at the time of the experiment. Calibration values are time-dependent observations, not permanent device specifications.

In [ ]:
backend_metadata = {}

if backend is not None:
    status = backend.status()
    backend_metadata = {
        "backend_name": backend.name,
        "num_qubits": backend.num_qubits,
        "operational": getattr(status, "operational", None),
        "pending_jobs": getattr(status, "pending_jobs", None),
        "backend_version": getattr(backend, "version", None),
    }

backend_metadata

## 5. Extract Calibration Data

This section attempts to extract commonly available backend properties. Property names can vary between IBM backend generations, so missing values are preserved as `None` rather than being invented.

In [ ]:
def safe_qubit_property(properties, qubit, parameter):
    try:
        value = properties.qubit_property(qubit, parameter)
        return value[0] if value else None
    except Exception:
        return None

calibration_rows = []

if backend is not None:
    try:
        properties = backend.properties()
    except Exception as exc:
        properties = None
        print("Backend properties unavailable:", exc)

    if properties is not None:
        for q in range(backend.num_qubits):
            calibration_rows.append({
                "backend": backend.name,
                "qubit": q,
                "T1_s": safe_qubit_property(properties, q, "T1"),
                "T2_s": safe_qubit_property(properties, q, "T2"),
                "readout_error": safe_qubit_property(properties, q, "readout_error"),
                "prob_meas0_prep1": safe_qubit_property(properties, q, "prob_meas0_prep1"),
                "prob_meas1_prep0": safe_qubit_property(properties, q, "prob_meas1_prep0"),
            })

calibration_df = pd.DataFrame(calibration_rows)

print("Calibration rows:", len(calibration_df))
calibration_df.head()

## 6. Save Calibration Snapshot

The calibration table is saved independently because it is a key input to the later adaptive QEM selector.

In [ ]:
calibration_path = CALIBRATION_DIR / f"{BACKEND_NAME}_calibration.csv"
metadata_path = CALIBRATION_DIR / f"{BACKEND_NAME}_metadata.json"

if not calibration_df.empty:
    calibration_df.to_csv(calibration_path, index=False)

metadata_path.write_text(
    json.dumps(backend_metadata, indent=2, default=str),
    encoding="utf-8"
)

print("Saved calibration snapshot:", calibration_path)
print("Saved backend metadata  :", metadata_path)

## 7. Transpile the Benchmark Circuits

The transpiled circuit is the hardware-oriented representation used for execution. Post-transpilation depth and two-qubit-gate counts are therefore important hardware metrics.

In [ ]:
transpiled_circuits = {}
transpile_rows = []

if backend is not None:
    for name, qc in benchmarks.items():
        try:
            tqc = transpile(
                qc,
                backend=backend,
                optimization_level=OPTIMIZATION_LEVEL,
                seed_transpiler=SEED_TRANSPILER,
            )

            transpiled_circuits[name] = tqc
            ops = tqc.count_ops()

            transpile_rows.append({
                "circuit": name,
                "original_qubits": qc.num_qubits,
                "original_depth": qc.depth(),
                "original_size": qc.size(),
                "transpiled_qubits": tqc.num_qubits,
                "transpiled_depth": tqc.depth(),
                "transpiled_size": tqc.size(),
                "sx": ops.get("sx", 0),
                "rz": ops.get("rz", 0),
                "x": ops.get("x", 0),
                "cx": ops.get("cx", 0),
                "cz": ops.get("cz", 0),
                "swap": ops.get("swap", 0),
                "measure": ops.get("measure", 0),
            })

        except Exception as exc:
            print(f"Transpilation failed for {name}: {exc}")

transpile_df = pd.DataFrame(transpile_rows)
transpile_df

## 8. Hardware Execution Function

The execution path is intentionally isolated in one function. If the installed IBM Runtime version requires a Runtime primitive rather than direct backend execution, update `hardware/execute_ibm.py` and keep this notebook's experimental structure unchanged.

In [ ]:
def execute_hardware_circuit(circuit, shots=SHOTS):
    if backend is None:
        raise RuntimeError("IBM backend is not available.")

    job = backend.run(circuit, shots=shots)
    return job

## 9. Submit Hardware Benchmark Runs

**Hardware execution is disabled by default.** Set `RUN_HARDWARE = True` only after authentication, backend selection, and transpilation have been verified. Hardware execution consumes IBM Quantum job resources.

In [ ]:
RUN_HARDWARE = False

hardware_jobs = {}

if RUN_HARDWARE:
    if backend is None:
        raise RuntimeError("Cannot execute: IBM backend is unavailable.")

    for name, tqc in transpiled_circuits.items():
        print(f"Submitting: {name}")
        job = execute_hardware_circuit(tqc, shots=SHOTS)
        hardware_jobs[name] = job
        print("Job ID:", job.job_id())
else:
    print("RUN_HARDWARE=False — no IBM hardware jobs were submitted.")

## 10. Collect Completed Hardware Results

Submission and result collection are separated so long-running jobs can be handled without accidentally resubmitting circuits.

In [ ]:
hardware_counts = {}
hardware_rows = []

if hardware_jobs:
    for name, job in hardware_jobs.items():
        print(f"Collecting: {name} | {job.job_id()}")

        result = job.result()
        counts = result.get_counts()
        hardware_counts[name] = counts

        tqc = transpiled_circuits[name]
        ops = tqc.count_ops()

        hardware_rows.append({
            "backend": backend.name,
            "circuit": name,
            "job_id": job.job_id(),
            "shots": SHOTS,
            "transpiled_qubits": tqc.num_qubits,
            "transpiled_depth": tqc.depth(),
            "transpiled_size": tqc.size(),
            "sx": ops.get("sx", 0),
            "rz": ops.get("rz", 0),
            "cx": ops.get("cx", 0),
            "cz": ops.get("cz", 0),
            "swap": ops.get("swap", 0),
            "observed_states": len(counts),
            "most_likely_state": max(counts, key=counts.get),
            "most_likely_probability": max(counts.values()) / SHOTS,
        })
else:
    print("No completed hardware jobs available in this notebook session.")

hardware_results_df = pd.DataFrame(hardware_rows)
hardware_results_df

## 11. Calculate Benchmark Success Probability

For deterministic benchmarks, the measured hardware success probability uses the same definitions as the ideal and noisy stages.

In [ ]:
def probability_of_states(counts, states, shots=SHOTS):
    return sum(counts.get(state, 0) for state in states) / shots

def hardware_success(name, counts):
    if name == "Bell_Phi_Plus":
        return probability_of_states(counts, ["00", "11"])

    if name.startswith("GHZ_"):
        n = int(name.split("_")[1])
        return probability_of_states(counts, ["0" * n, "1" * n])

    if name.startswith("Superdense_"):
        message = name.split("_")[1]
        return counts.get(message, 0) / SHOTS

    if name == "Grover_2Q":
        return counts.get("11", 0) / SHOTS

    return float("nan")

if not hardware_results_df.empty:
    hardware_results_df["benchmark_success_probability"] = [
        hardware_success(row["circuit"], hardware_counts[row["circuit"]])
        for _, row in hardware_results_df.iterrows()
    ]
    hardware_results_df["error_probability"] = (
        1 - hardware_results_df["benchmark_success_probability"]
    )

hardware_results_df

## 12. Save Hardware Data

Hardware results, transpilation metrics, backend metadata, and raw counts are stored separately. These files form the evidence base for the QEM comparison.

In [ ]:
transpile_path = DATA_DIR / f"{BACKEND_NAME}_transpilation_metrics.csv"
hardware_results_path = DATA_DIR / f"{BACKEND_NAME}_hardware_results.csv"
counts_path = DATA_DIR / f"{BACKEND_NAME}_raw_counts.json"
run_config_path = DATA_DIR / f"{BACKEND_NAME}_run_config.json"

if not transpile_df.empty:
    transpile_df.to_csv(transpile_path, index=False)

if not hardware_results_df.empty:
    hardware_results_df.to_csv(hardware_results_path, index=False)

counts_path.write_text(
    json.dumps(hardware_counts, indent=2),
    encoding="utf-8"
)

run_config_path.write_text(
    json.dumps({
        "backend": BACKEND_NAME,
        "shots": SHOTS,
        "optimization_level": OPTIMIZATION_LEVEL,
        "seed_transpiler": SEED_TRANSPILER,
        "hardware_execution_enabled": RUN_HARDWARE,
    }, indent=2),
    encoding="utf-8"
)

print("Saved hardware-stage files:")
for path in [transpile_path, hardware_results_path, counts_path, run_config_path]:
    print("-", path)

## 13. Hardware Benchmark Checklist

Before using these data in the paper, verify that every executed circuit has a valid job ID, shot count, backend name, calibration snapshot, transpiled metrics, and raw counts.

In [ ]:
if hardware_results_df.empty:
    print("Hardware execution has not been collected yet.")
    print("Set RUN_HARDWARE=True, submit jobs, wait for completion, and rerun the collection cells.")
else:
    required = [
        "backend",
        "circuit",
        "job_id",
        "shots",
        "transpiled_depth",
        "transpiled_size",
    ]

    missing = [column for column in required if column not in hardware_results_df.columns]

    print("Required columns missing:", missing)
    print("Completed hardware records:", len(hardware_results_df))
    print("Unique job IDs:", hardware_results_df["job_id"].nunique())

## Next Step

**`05_qem_analysis.ipynb`** will compare raw IBM hardware results with readout-error mitigation and zero-noise extrapolation, calculate fidelity/error improvement and execution overhead, and prepare the data required by the adaptive QEM selector.